[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/themintlab/Znet/blob/main/znet/examples/Extruded_dimension.ipynb)

In [ ]:
# Install Znet if running in Google Colab
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/themintlab/Znet.git

In [1]:
import torch
from znet.core import FactorNode, SignalNode, ConstantNode, LeafNode
from znet.transforms import finalize, legendre_transform
import plotly.graph_objects as go

In [ ]:
R = 8.314
RT = R*300

In [ ]:
n1 = ConstantNode(-1)
mu1 = TPowNode([n1, SignalNode(1)])
mu2 = TPowNode([n1, SignalNode(2)])
Va = SignalNode(3)
RT0 = ConstantNode(RT)
Vb = TProdNode([ConstantNode(1), TPowNode([n1, Va])])

In [ ]:
mu1A = TProdNode([RT0, mu1])
mu2A = mu2
mu1B = mu1
mu2B = TProdNode([RT0, mu2])

In [ ]:
phaseA = FactorNode([mu1A, mu2A], beta_factor = 8.314)
phaseA = TPowNode([Va, phaseA])
phaseB = FactorNode([mu1B, mu2B], beta_factor = 8.314)
phaseB = TPowNode([Vb, phaseB])
system = TProdNode([phaseA, phaseB])#  #FactorNode([phaseA, phaseB], beta_factor = 1e-8*8.314)

In [ ]:
phaseAb = vmap(phaseA)
phaseBb = vmap(phaseB)
systemb = vmap(system)

# Grand potential plot

In [7]:
T_val = torch.tensor(298.15)
grid_size = 500
mu1_range = torch.linspace(-10 * RT, 10 * RT, steps=grid_size)
V_range = torch.linspace(0., 1., steps=grid_size)

N_mu1 = grid_size
N_V = grid_size

In [8]:
MU1_grid, V_grid = torch.meshgrid(mu1_range, V_range, indexing="ij")

MU1_flat = MU1_grid.flatten()
V_flat = V_grid.flatten()

T_flat = T_val.expand_as(MU1_flat)
MU2_flat = -MU1_flat

input_tensor = torch.stack([T_flat, MU1_flat, MU2_flat, V_flat], dim=-1)

In [9]:
z_A_flat = phaseAb(input_tensor)
z_B_flat = phaseBb(input_tensor)
z_sys_flat = systemb(input_tensor)

x_np = MU1_grid.cpu().numpy()
y_np = V_grid.cpu().numpy()
z_A = z_A_flat.detach().cpu().numpy().reshape(N_mu1, N_V)
z_B = z_B_flat.detach().cpu().numpy().reshape(N_mu1, N_V)
z_sys = z_sys_flat.detach().cpu().numpy().reshape(N_mu1, N_V)

In [10]:
# 3. Build the Plotly Figure with all three surfaces
fig = go.Figure()
# Add Phase A (Blue)
fig.add_trace(go.Surface(
    x=x_np,
    y=y_np,
    z=z_A,
    colorscale="Blues",
    name="Phase A",
    showscale=False,
    showlegend=True
))
# Add Phase B (Red)
fig.add_trace(go.Surface(
    x=x_np,
    y=y_np,
    z=z_B,
    colorscale="Reds",
    name="Phase B",
    showscale=False,
    showlegend=True
))
# Add System (Viridis)
fig.add_trace(go.Surface(
    x=x_np,
    y=y_np,
    z=z_sys,
    colorscale="Viridis",
    name="System",
    showscale=False,
    showlegend=True
))
# 4. Set Axis Labels and Layout
fig.update_layout(
    title="Grand Potential Surfaces (Omega)",
    scene=dict(
        xaxis=dict(title="Chemical Potential (mu_1)"),
        yaxis=dict(title="Volume (V)"),
        zaxis=dict(title="Grand Potential (Omega)")
    ),
    width=1000,
    height=800,
    margin=dict(l=65, r=50, b=65, t=90)
)
fig.show()

# Free energy plot

In [ ]:
fA = vmap(legendre_transform(phaseA, [1,2]))
fAd, cAd = fA(input_tensor)

fB = vmap(legendre_transform(phaseB, [1,2]))
fBd, cBd = fB(input_tensor)

fsys = vmap(legendre_transform(system, [1,2]))
fsysd, csysd = fsys(input_tensor)

In [12]:


x_A = cAd[:, 1].detach().cpu().numpy().reshape(N_mu1, N_V)
y_A = cAd[:, 3].detach().cpu().numpy().reshape(N_mu1, N_V) # Va
z_A = -fAd.detach().cpu().numpy().reshape(N_mu1, N_V)

x_B = cBd[:, 1].detach().cpu().numpy().reshape(N_mu1, N_V)
y_B = cBd[:, 3].detach().cpu().numpy().reshape(N_mu1, N_V) # Va
z_B = -fBd.detach().cpu().numpy().reshape(N_mu1, N_V)

x_sys = csysd[:, 1].detach().cpu().numpy().reshape(N_mu1, N_V)
y_sys = csysd[:, 3].detach().cpu().numpy().reshape(N_mu1, N_V) # Va
z_sys = -fsysd.detach().cpu().numpy().reshape(N_mu1, N_V)

In [13]:
# 3. Build the Plotly Figure
fig = go.Figure()
# Add Phase A (Blue colorscale)
fig.add_trace(go.Surface(
    x=x_A, y=y_A, z=z_A,
    colorscale="Blues",
    name="Phase A",
    showscale=False,
    showlegend=True
))
# Add Phase B (Red colorscale)
fig.add_trace(go.Surface(
    x=x_B, y=y_B, z=z_B,
    colorscale="Reds",
    name="Phase B",
    showscale=False,
    showlegend=True
))
#Add System (Viridis colorscale)
fig.add_trace(go.Surface(
    x=x_sys, y=y_sys, z=z_sys,
    colorscale="Viridis",
    name="System",
    showscale=False,
    showlegend=True
))


# 4. Set Axis Labels and Layout
fig.update_layout(
    scene=dict(
        xaxis=dict(title="Mole fraction"),
        yaxis=dict(title="Phase fraction"),
        zaxis=dict(title="Free Energy")
    ),
    width=1000,
    height=800,
    margin=dict(l=65, r=50, b=65, t=90)
)
fig.show()